# KV-Cache

## 什么是KV-Cache
KV-Cache是一种常见的提高大模型推理性能的技术，使用上一次推理的`Key`，`Value`缓存来提高推理性能，并降低端到端的延迟，并且不影响人和准确性。几乎所有自回归模型都内置了KV-Cache，理解KV-Cache有助于更深刻地认识Transformer中注意力机制的工作方式。

## 为什么需要KV-Cache？
首先，让我们回顾一下`Attention`计算公式：
$$Atentino(\mathit{Q},\mathit{K},\mathit{V})=softmax(\frac{\mathit{Q}\mathit{K}^T}{\sqrt{d_k}})\mathit{V}$$

> 注意，在注意力计算过程中，每一个token：$T_i$都要与矩阵$Q,K,V$计算得到三个向量$q_n^i,k_n^i,v_n^i$，然后再进行注意力计算。强调：每一次进行注意力层计算时，每个token都要这么计算，并且，每个token的$q_n$都要与其他token的$k_n,v_n$进行计算。

我们再来看看模型在推理过程中，进行一次前向推理的过程：
- 输入序列为n个token $\{T_1, \cdots , T_i , \cdots , T_n\}$
- 在Embedding阶段，每个token被embedding成$D$维的向量，因此Embedding层的输出维度为`[batch_size, token_length, embedding_dim]`。这里假设每个token被embedding之后的向量为$x_i^l i\in[0,...,n-1],l\in[0,...,layers]$，那么，输出的向量矩阵为：$\{x_1^0, \cdots , x_i^0 , \cdots , x_n^0\}$
- 在接下来，每一层（Attention&MLP）的计算过程中，矩阵的维度保持不变。
- 最后则是根据计算结果生成下一个token，得到新的token：$T_{n+1}$。
- 再继续生成下一个token时，将$T_{n+1}$加入到输入序列中，得到$\{T_1, \cdots , T_i , \cdots , T_n， T_{n+1}\}$。重复前面的计算过程。

![alt text](./_img/forwar_one.png)

> 在这个过程中，需要注意的是模型每次在计算下一个token时，都会将整个序列重新输入。

![alt text](./_img/forwar_2.png)

可以运行下面这段代码，直观感受一下不断生成下一个词的过程。

In [ ]:
import torch

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
# torch.manual_seed(0)

class Sampler:
    def __init__(self , model_name : str ='gpt2-medium') -> None:

        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name).to("cpu").to(self.device)

    def encode(self, text):
        return self.tokenizer.encode(text, return_tensors='pt').to(self.device)

    def decode(self, ids):
        return self.tokenizer.decode(ids)

    def get_next_token_prob(self, input_ids: torch.Tensor):
        with torch.no_grad():
            logits = self.model(input_ids=input_ids).logits
        logits = logits[0, -1, :]
        return logits
    
class GreedySampler(Sampler):
    def __call__(self, prompt, max_new_tokens=10):
        predictions = []
        result = prompt
        # generate until max_len
        for i in range(max_new_tokens):
            
            print(f"step {i} input: {result}")
            input_ids = self.encode(result)
            next_token_probs = self.get_next_token_prob(input_ids=input_ids)
            
            # choose the token with the highest probability
            id = torch.argmax(next_token_probs, dim=-1).item()
            # convert to token and add new token to text
            result += self.decode(id)
            
            predictions.append(next_token_probs[id].item())

        return result


到这里，就已经可以看出，当模型在计算序列$\{T_1, \cdots , T_i , \cdots , T_n， T_{n+1}\}$时，其中$\{T_1, \cdots , T_i , \cdots , T_n\}$的$q_n,k_n,v_n$已经被计算过了，实际上需要计算的是$T_{n+1}$与前面token的关系。所以，就可以把前面的token的$k_n,v_n$缓存起来，至于为什么不缓存$q_n$，后面回讲到。

可以通过下面这段代码来验证这一点：

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2Model
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2Model.from_pretrained('gpt2')

# text: "The quick brown fox jumps over the lazy"
tokens = [[464, 2068, 7586, 21831, 18045, 625, 262, 16931]]
input_n = torch.tensor(tokens)
output_n = model(input_ids=input_n, output_hidden_states=True)

# text: " dog"
tokens[0].append(3290)
input_n_plus_1 = torch.tensor(tokens)
output_n_plus_1 = model(input_ids=input_n_plus_1, output_hidden_states=True)

for i, (hidden_n, hidden_n_plus_1) in enumerate(zip(output_n.hidden_states, output_n_plus_1.hidden_states)):
    print(f"layer {i}, max difference {(hidden_n - hidden_n_plus_1[:, :-1, :]).abs().max().item()}")
    assert torch.allclose(hidden_n, hidden_n_plus_1[:, :-1, :], atol=1e-4)

## KVCache工作原理
首先，需要保证理解下面这图的`scaled dot-product attention`计算过程。
<img src="./_img/att_cal.gif"/>

原理其实非常简单：
1. 假设模型架构有`n`个`transformer`层，采用多头注意力机制，那么每一层的每个头都将单独维护一个`KVCache`。如下面这段代码所示：

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()

        ... 

        self.cache_k = torch.zeros((args.max_batch_size, args.max_seq_len, self.n_kv_heads, self.head_dim))
        self.cache_v = torch.zeros((args.max_batch_size, args.max_seq_len, self.n_kv_heads, self.head_dim))


2. 在前向计算过程中，将缓存的`KV`取出，和最新`token`的`k`,`v`拼接到一起，参与注意力机制计算。

如下面这段代码所示：

In [ ]:
def forward(
    self,
    x: torch.Tensor,
    start_pos: int,
    freqs_complex: torch.Tensor
):
    ... 

    # Input shape : (B, 1, Dim)
    # xk of shape (B, 1, H_KV, Head_Dim)
    # xv of shape (B, 1, H_KV, Head_Dim)
    # Replace the entry in the cache
    self.cache_k[:batch_size, start_pos : start_pos + seq_len] = xk
    self.cache_v[:batch_size, start_pos : start_pos + seq_len] = xv

    # (B, Seq_Len_KV, H_KV, Head_Dim)
    keys = self.cache_k[:batch_size, : start_pos + seq_len]
    # (B, Seq_Len_KV, H_KV, Head_Dim)
    values = self.cache_v[:batch_size, : start_pos + seq_len]


数学上，假设生成的标记位于 $i^{th}$ transformer层。它表示为以下 $t_i \in R^{b×1×h}$ 。 $i^{th}$ 变换器内部的计算分为两个部分：更新 `KV` 缓存和计算 $t^{i+1}$ 。

更新`K,V`缓存的公式为：
![alt text](./_img/f1.png)

计算下一个token的公式为：
![alt text](./_img/f2.png)

下图是一个由12个注意力头，使用`KVCache`计算注意力的过程：
![alt text](./_img/KVCache_work.png)
![alt text](./_img/kvcache_work1.gif)

下面这段代码是使用`gpt2`生成1000个新token时，用和不用KVCache的速度差异

In [ ]:
import numpy as np
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

for use_cache in (True, False):
    times = []
    for _ in range(10):  # measuring 10 generations
        start = time.time()
        model.generate(**tokenizer("What is KV caching?", return_tensors="pt").to(device), use_cache=use_cache, max_new_tokens=1000)
        times.append(time.time() - start)
    print(f"{'with' if use_cache else 'without'} KV caching: {round(np.mean(times), 3)} +- {round(np.std(times), 3)} seconds")

## 计算量分析 FLOPs 比较
关于带和不带kvcache的计算量比较，看这篇：[FLOPs comparison of vanilla Transformer and Transformer with KV Cache](https://r4j4n.github.io/blogs/posts/kv/#flops-comparison-of-vanilla-transformer-and-transformer-with-kv-cache)

$memory\_usage\_per\_token$

## KV 缓存使用多少内存？
让我们考虑一个 13B 参数的 OPT 模型，每个token所占用的内存为:
$$memory\_usage\_per\_token = num\_vectors * hidden\_state_size * num\_layers * \text{precision (bytes)} \\ = 2 * 5120 * 40 * 2 = 800\; \text{KB}$$

num_vectors指的是键和值向量。

在OPT模型中，输入序列的最大长度为2048个token，所以，需要的缓存为$800 * 2048 \approx 1.6\; \text{GB}$

![alt text](./_img/kvmap.png)

## KVCache在LLaMA模型中的应用


## 参考资料：
1. [Transformers Optimization: Part 1 - KV Cache](https://r4j4n.github.io/blogs/posts/kv/)
2. [What is the KV cache?](https://mett29.github.io/posts/kv-cache/)
3. [一文读懂KVCache](https://zhuanlan.zhihu.com/p/686183300)